# Pipeline Effect Size Analysis
## Linear Mixed-Effects Model for Factorial Deconvolution Pipeline

**Design**: 3 (Classifiers) × 4 (Labeling schemes) × 5 (Deconvolvers) × 4 (Calibrators) = 240 combinations  
**Observations**: 100,000 pseudobulks with identical compositions across all combinations  
**Outcome**: MSE between predicted and true cell-type proportions  
**Random effect**: Pseudobulk index (crossed — same composition seen by all 240 pipelines)

---
## 0. Setup

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from scipy import stats
from scipy.stats import f as f_dist
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
rng  = np.random.default_rng(SEED)

# Plot style
plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
sns.set_palette('colorblind')

print('All packages loaded.')

All packages loaded.


---
## 1. Data

### 1b. Load real data (use instead of §1a)

Your dataframe must have one row per (combination, pseudobulk) with these columns:

| Column | Type | Description |
|---|---|---|
| `MSE` | float | Raw MSE |
| `logMSE` | float | log(MSE) — add with `df['logMSE'] = np.log(df['MSE'])` |
| `Classifier` | category | C1, C2, C3 |
| `Labeling` | category | L1 … L4 |
| `Deconvolver` | category | D1 … D5 |
| `Calibrator` | category | K1 … K4 |
| `pb_index` | int | 0-based pseudobulk index (same index = same composition) |

In [3]:
# Define paths
CLASSIFIERS = ["dismir", "methylbert", "lookup"]
LABELING_SCHEMES = ["soft_w_pool", "hard_diagbg", "hard_top156"]#"soft_wo_pool", 
DECONVOLVERS = ["xgb", "mlp", "swn", "nnls", "psls"]
CALIBRATORS = ["none", "lin_clip0_normalize", "lin_simplex", "vector_scaling"]
PATH_TO_PER_SAMPLE_METRICS = {
    "dismir":{
        "soft_w_pool": "/staging/leuven/stg_00118/methylDL/experiments/ExtendedProportions/pseudobulk/Dismir_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_soft_labels_pooled_jakkard_no_data_leak_d041/pseudobulk/fitted_deconvolvers_unifrorm_multinomial_all_top156_features/callibration",
        "hard_top156": "/staging/leuven/stg_00118/methylDL/experiments/ExtendedProportions/pseudobulk/Dismir_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_hard_labels/pseudobulk/fitted_deconvolvers_unifrorm_multinomial_all_top156_features/callibration",
        # "soft_wo_pool":"/staging/leuven/stg_00118/methylDL/experiments/ExtendedProportions/pseudobulk/Dismir_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_soft_labels_NO_POOLING/pseudobulk/fitted_deconvolvers_unifrorm_multinomial_all_top156_features/callibration",
        "hard_diagbg":"/staging/leuven/stg_00118/methylDL/experiments/ExtendedProportions/pseudobulk/Dismir_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_hard_labels/pseudobulk/fitted_deconvolvers_unifrorm_multinomial_all_diagrej_features/callibration"
    },
    "methylbert":{
        "soft_w_pool": "/staging/leuven/stg_00118/methylDL/experiments/ExtendedProportions/pseudobulk/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_soft_labels_pooled_jakkard_no_data_leak_d041/pseudobulk/fitted_deconvolvers_unifrorm_multinomial_all_top156features/callibration",
        "hard_top156": "/staging/leuven/stg_00118/methylDL/experiments/ExtendedProportions/pseudobulk/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_hard_labels_minibatch_balanced/pseudobulk/fitted_deconvolvers_unifrorm_multinomial_all_top156_features/callibration",
        # "soft_wo_pool":"/staging/leuven/stg_00118/methylDL/experiments/ExtendedProportions/pseudobulk/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_no_data_leak_postfiltered_min_length_50_soft_labels_NO_POOLING_jakkard/pseudobulk/fitted_deconvolvers_unifrorm_multinomial_all_top156_features/callibration",
        "hard_diagbg":"/staging/leuven/stg_00118/methylDL/experiments/ExtendedProportions/pseudobulk/methylBert_Loyfer_205files_U25_attentionClassifier_hg38_dmr_ctype_label_mincpg_4_minlen_10_postfiltered_min_length_50_hard_labels_minibatch_balanced/pseudobulk/fitted_deconvolvers_unifrorm_multinomial_all_diagrej_features/callibration"
    },
    "lookup":{
        "soft_w_pool": "/staging/leuven/stg_00118/methylDL/experiments/ExtendedProportions/pseudobulk/LookupClassifier_SoftLabels_Ours/pseudobulk/fitted_deconvolvers_unifrorm_multinomial_all_top156features/callibration",
        "hard_top156": "/staging/leuven/stg_00118/methylDL/experiments/ExtendedProportions/pseudobulk/LookupClassifier_HardLabels/pseudobulk/fitted_deconvolvers_unifrorm_multinomial_all_top156_features/callibration",
        # "soft_wo_pool":"/staging/leuven/stg_00118/methylDL/experiments/ExtendedProportions/pseudobulk/LookupClassifier_SoftLabels_WITHOUTPOOLING/pseudobulk/fitted_deconvolvers_unifrorm_multinomial_all_top156_features/callibration",
        "hard_diagbg":"/staging/leuven/stg_00118/methylDL/experiments/ExtendedProportions/pseudobulk/LookupClassifier_HardLabels/pseudobulk/fitted_deconvolvers_unifrorm_multinomial_all_diagrej_features/callibration"
    }
}
PATH_TO_PER_SAMPLE_METRICS = {
    model: {
        labeling: Path(path) / "per_sample_metrics.csv"
        for labeling, path in labeling_dict.items()
    }
    for model, labeling_dict in PATH_TO_PER_SAMPLE_METRICS.items()
    if model in CLASSIFIERS
}

In [4]:
# load the data
data_per_classifier_labeling = {}
for model, labeling_dict in PATH_TO_PER_SAMPLE_METRICS.items():
    data_per_classifier_labeling[model] = {}
    for labeling, path in labeling_dict.items():
        df = pd.read_csv(path)
        df.drop(columns=['mae', "kl"], inplace=True)
        data_per_classifier_labeling[model][labeling] = df

The issue is that the indexes of samples inside the pseudobulks are not the same accross pseudobulks: the pseudobulks have the same proportions but not necessarily in the same order.

This means we also have to load the target proportions of the pseudobulks, and reorder the samples in each pseudobulk to match the same order of samples in the target proportions. This way, the same pseudobulk index will correspond to the same composition across all pipelines.

In [5]:
# load the target proportions of the pseudobulks
target_proportions_per_classifier_labeling = {}
for model, labeling_dict in PATH_TO_PER_SAMPLE_METRICS.items():
    target_proportions_per_classifier_labeling[model] = {}
    for labeling, path in labeling_dict.items():
        target_proportions_per_classifier_labeling[model][labeling] = np.load(path.parent / "mlp_calibrators_and_predictions" / "uncalibrated_predictions.npz")["test_target"]

In [6]:
# We sort the target proportions by alphabetical order 
def rows_lex_argsort(a):
    a = np.asarray(a)
    if a.ndim != 2:
        raise ValueError("Input must be 2D")
    # lexsort takes keys from least- to most-significant, so pass columns reversed
    keys = tuple(a[:, i] for i in range(a.shape[1] - 1, -1, -1))
    return np.lexsort(keys)

# Usage on a single array
arr = np.array([[0, 2, 1],
                [0, 1, 5],
                [1, 0, 0]])
order = rows_lex_argsort(arr)   # indices that lexicographically sort rows (cols left→right)
sorted_rows = arr[order]

# Apply to your dict of target proportions
lex_idx = {}
for model, lab_dict in target_proportions_per_classifier_labeling.items():
    lex_idx[model] = {}
    for labeling, targ in lab_dict.items():
        lex_idx[model][labeling] = rows_lex_argsort(targ)

In [ ]:
sorted_target_proportions = target_proportions_per_classifier_labeling["methylbert"]["soft_w_pool"][lex_idx["methylbert"]["soft_w_pool"]]

Check that the sorted proportions are equal:

In [ ]:
(np.all(target_proportions_per_classifier_labeling["methylbert"]["soft_w_pool"][lex_idx["methylbert"]["soft_w_pool"]] == target_proportions_per_classifier_labeling["dismir"]["hard_top156"][lex_idx["dismir"]["hard_top156"]]),
np.all(target_proportions_per_classifier_labeling["methylbert"]["soft_w_pool"][lex_idx["methylbert"]["soft_w_pool"]] == target_proportions_per_classifier_labeling["methylbert"]["hard_top156"][lex_idx["methylbert"]["hard_top156"]]),
np.all(target_proportions_per_classifier_labeling["dismir"]["soft_w_pool"][lex_idx["dismir"]["soft_w_pool"]] == target_proportions_per_classifier_labeling["dismir"]["hard_top156"][lex_idx["dismir"]["hard_top156"]]),)

In [7]:
# add a column with common pseudobulk index
for model, lab_dict in data_per_classifier_labeling.items():
    for labeling, df in lab_dict.items():
        df['pb_index'] = df['sample_index'].map(lex_idx[model][labeling].__getitem__)

In [8]:
# agregate in a single dataframe
dataframes_to_concat = []
for model, lab_dict in data_per_classifier_labeling.items():
    for labeling, df in lab_dict.items():
        df = df.copy()
        df.drop(columns=['sample_index'], inplace=True)
        df['Classifier'] = model
        df['Labeling'] = labeling
        df.sort_values('pb_index', inplace=True) 
        dataframes_to_concat.append(df)
del df

In [9]:
final_df = pd.concat(dataframes_to_concat, axis=0, ignore_index=True)
final_df.rename(columns={'deconvolver': 'Deconvolver', "calibration_method":"Calibrator", "mse": "MSE"}, inplace=True)
for col_name in ["Classifier", "Labeling", "Deconvolver", "Calibrator"]:
    final_df[col_name] = final_df[col_name].astype('category')
final_df["logMSE"] = np.log(final_df["MSE"])

In [10]:
final_df.head()

,Deconvolver,Calibrator,MSE,pb_index,Classifier,Labeling,logMSE
0,swn,linear_clip0_normalize,0.000134,0,dismir,soft_w_pool,-8.918050
1,xgb,linear_simplex_projection,0.000034,0,dismir,soft_w_pool,-10.301187
2,nnls,linear_clip0_normalize,0.000240,0,dismir,soft_w_pool,-8.336736
3,psls,vector_scaling,0.000564,0,dismir,soft_w_pool,-7.480023
4,xgb,uncalibrated,0.000044,0,dismir,soft_w_pool,-10.040279


In [11]:
# OPTIONAL: save the final dataframe to csv
final_df.to_csv("/staging/leuven/stg_00118/methylDL/experiments/ExtendedProportions/pseudobulk/effect_sizes_analysis/mse_per_classifier_labeling_deconvolver_calibrator_pseudobulk.csv", index=False)

In [ ]:
for col_name in ["Classifier", "Labeling", "Deconvolver", "Calibrator"]:
    print(final_df[col_name].cat.categories)

In [ ]:
N_COMBOS = len(final_df["Classifier"].cat.categories) * len(final_df["Labeling"].cat.categories) * len(final_df["Deconvolver"].cat.categories) * len(final_df["Calibrator"].cat.categories)

### 1c. Subsample for tractable fitting

24M rows is expensive. We subsample pseudobulks while keeping **all 240 combinations** — the design stays fully crossed.

In [ ]:
N_PB_SAMPLE = 10_000   # ← increase for more precision

sampled_pbs = rng.choice(final_df['pb_index'].unique(), size=N_PB_SAMPLE, replace=False)
df_sub = final_df[final_df['pb_index'].isin(sampled_pbs)].copy()

# Re-encode pb_index as string category for statsmodels groups
df_sub['pb_id'] = df_sub['pb_index'].astype(str)

# recompute logMSE with clipping
df_sub['logMSE'] = np.log(df_sub['MSE'])
lower_clip = df_sub["logMSE"].quantile(0.01)
df_sub['logMSE'] = df_sub['logMSE'].clip(lower=lower_clip)

# Ensure categorical dtypes with explicit reference levels
for col, ref in [('Classifier','lookup'), ('Labeling','hard_top156'),
                 ('Deconvolver','nnls'), ('Calibrator','uncalibrated')]:
    df_sub[col] = pd.Categorical(df_sub[col], categories=sorted(df_sub[col].unique()))

print(f'Subsampled dataset: {df_sub.shape}  ({N_PB_SAMPLE:,} pseudobulks x {N_COMBOS} combos)')
df_sub[['Classifier','Labeling','Deconvolver','Calibrator','pb_id','logMSE']].head()

In [ ]:
df_sub.head()

---
## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
factors = ['Classifier', 'Labeling', 'Deconvolver', 'Calibrator']

# ── Raw MSE distributions per factor ────────────────────────────────────────
for ax, factor in zip(axes[0], factors):
    means = df_sub.groupby(factor, observed=True)['MSE'].mean().reset_index()
    sns.boxplot(data=df_sub, x=factor, y='MSE', ax=ax,
                showfliers=False, width=0.5)
    ax.set_title(f'MSE by {factor}', fontweight='bold')
    ax.set_xlabel('')

# ── log-MSE distributions per factor ────────────────────────────────────────
for ax, factor in zip(axes[1], factors):
    sns.boxplot(data=df_sub, x=factor, y='logMSE', ax=ax,
                showfliers=False, width=0.5, color='steelblue')
    ax.set_title(f'log(MSE) by {factor}', fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Marginal distributions by pipeline factor', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# ── Normality check on logMSE ────────────────────────────────────────────────
sample_vals = df_sub['logMSE'].sample(5000, random_state=SEED)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.hist(sample_vals, bins=60, color='steelblue', edgecolor='white', linewidth=0.4)#, range=(-16, sample_vals.max()))
ax1.set_title('log(MSE) — histogram (5k sample)')
ax1.set_xlabel('log(MSE)')
stats.probplot(sample_vals, dist='norm', plot=ax2)
ax2.set_title('Q-Q plot of log(MSE)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Combination-level mean heatmap (Deconvolver × Calibrator) ────────────────
pivot = (df_sub.groupby(['Deconvolver','Calibrator'], observed=True)['logMSE']
               .mean()
               .unstack('Calibrator'))

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Mean log(MSE)'})
ax.set_title('Mean log(MSE): Deconvolver × Calibrator\n(averaged over Classifier & Labeling)',
             fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Linear Mixed-Effects Models

We build models in steps, from null to full, comparing them with likelihood ratio tests.

> **Model structure**  
> - **Outcome**: `logMSE` (log-transformation yields approximately normal residuals)  
> - **Fixed effects**: Classifier, Labeling, Deconvolver, Calibrator (+ interactions)  
> - **Random effect**: `(1 | pb_id)` — crossed pseudobulk intercept absorbing composition difficulty  
> - **Estimator**: REML for final models; ML for likelihood-ratio model comparison

In [ ]:
def fit_lmm(formula_fixed, data, reml=True):
    """
    Fit LMM with crossed random intercept on pb_id.
    Returns the fitted MixedLMResults object.
    """
    model = smf.mixedlm(
        formula   = formula_fixed,
        data      = data,
        groups    = data['pb_id'],    # crossed random intercept
        re_formula= '~1',
    )
    return model.fit(reml=reml, method='lbfgs', maxiter=500)

def fit_ols(formula_fixed, data):
    """
    Fit OLS regression with fixed effects only.
    Returns the fitted RegressionResults object.
    """
    model = smf.ols(formula=formula_fixed, data=data)
    return model.fit()

def lrt(fit_null, fit_full):
    """
    Likelihood ratio test between two ML-fitted models.
    Both must be fitted with reml=False.
    """
    chi2 = 2 * (fit_full.llf - fit_null.llf)
    df   = fit_full.df_modelwc - fit_null.df_modelwc
    p    = 1 - stats.chi2.cdf(chi2, df=max(df, 1))
    return chi2, int(df), p


print('Helper functions defined.')

In [ ]:
# ── Step 1: Null model (random effect only) ───────────────────────────────────
print('Fitting null model...')
m0 = fit_lmm('logMSE ~ 1', df_sub, reml=True)

sigma2_pb       = float(m0.cov_re.iloc[0, 0])   # variance of random intercept
sigma2_resid_m0 = float(m0.scale)                # residual variance
icc             = sigma2_pb / (sigma2_pb + sigma2_resid_m0)

print(f'\n── Null model variance components ──────────────────')
print(f'  σ²(pseudobulk / composition) : {sigma2_pb:.5f}')
print(f'  σ²(residual)                 : {sigma2_resid_m0:.5f}')
print(f'  ICC (composition difficulty) : {icc:.3f}')
print(f'\n  ICC = {icc:.1%} of residual variance is composition-level.')
print(f'  → Mixed model is {"strongly" if icc > 0.3 else "moderately" if icc > 0.1 else "marginally"} justified.')

In [ ]:
# ── Step 2: Main effects model (OLS) ────────────────────────────────────────────────
print('Fitting main effects model (OLS)...')
formula_main = 'logMSE ~ Classifier + Labeling + Deconvolver + Calibrator'
m_main_ols = fit_ols(formula_main, df_sub)
print(m_main_ols.summary())

In [ ]:
pw = m_main_ols.t_test_pairwise("Labeling")
res = pw.result_frame.copy()

# Keep and rename the useful columns
res = res[['coef', 'std err', 't', 'P>|t|', 'pvalue-hs', 'reject-hs']].copy()
res.columns = ['Δ log(MSE)', 'SE', 't', 'p (raw)', 'p (Holm)', 'sig (Holm)']
res.index.name = 'Comparison'

# Add significance stars based on Holm-corrected p
def sig_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return 'ns'

res['stars'] = res['p (Holm)'].map(sig_stars)

res.style \
    .format({'Δ log(MSE)': '{:+.4f}', 'SE': '{:.4f}', 't': '{:+.2f}',
             'p (raw)': '{:.4f}', 'p (Holm)': '{:.4f}'}) \
    .background_gradient(subset=['Δ log(MSE)'], cmap='RdYlGn_r', vmin=-0.5, vmax=0.5) \
    .applymap(lambda v: 'color: green; font-weight: bold' if v == '***' else
                        'color: orange' if v in ('**', '*') else
                        'color: grey', subset=['stars']) \
    .set_caption('Pairwise Labeling comparisons — OLS, Holm-corrected')

In [ ]:
# ── Step 2: Main effects model ────────────────────────────────────────────────
print('Fitting main effects model (REML)...')
formula_main = 'logMSE ~ Classifier + Labeling + Deconvolver + Calibrator'
m_main = fit_lmm(formula_main, df_sub, reml=True)
print(m_main.summary())

In [ ]:
print("logMSE stats:")
print(df_sub['logMSE'].describe())
print("\nMSE stats:")
print(df_sub['MSE'].describe())

In [ ]:
# ── Step 3: Add all two-way interactions ──────────────────────────────────────
print('Fitting full two-way interaction model (REML)...')
formula_2way = ('logMSE ~ Classifier * Labeling '
                '+ Classifier * Deconvolver '
                '+ Classifier * Calibrator '
                '+ Labeling   * Deconvolver '
                '+ Labeling   * Calibrator '
                '+ Deconvolver * Calibrator')
m_2way = fit_lmm(formula_2way, df_sub, reml=True)
print(m_2way.summary())

In [ ]:
# ── Step 4: Likelihood ratio tests (refit with ML for fair comparison) ────────
print('Refitting with ML for LRT...')
m0_ml    = fit_lmm('logMSE ~ 1',         df_sub, reml=False)
m_main_ml= fit_lmm(formula_main,         df_sub, reml=False)
m_2way_ml= fit_lmm(formula_2way,         df_sub, reml=False)

chi2_main, df_main, p_main = lrt(m0_ml,     m_main_ml)
chi2_2way, df_2way, p_2way = lrt(m_main_ml, m_2way_ml)

print(f'\n── Likelihood Ratio Tests ─────────────────────────────────────────────')
print(f'  Null → Main effects  : χ²({df_main}) = {chi2_main:.1f},  p = {p_main:.2e}')
print(f'  Main → +Interactions : χ²({df_2way}) = {chi2_2way:.1f},  p = {p_2way:.2e}')
print(f'\n  AIC null    : {m0_ml.aic:.1f}')
print(f'  AIC main    : {m_main_ml.aic:.1f}  (Δ = {m0_ml.aic - m_main_ml.aic:.1f})')
print(f'  AIC 2-way   : {m_2way_ml.aic:.1f}  (Δ = {m_main_ml.aic - m_2way_ml.aic:.1f})')

---
## 4. Effect Size Decomposition

We compute **ω² (omega-squared)** for each factor and interaction — the proportion of total variance attributable to each effect, corrected for the degrees of freedom consumed.

In [ ]:
def compute_effect_sizes(df_data, formula_full, outcome='logMSE'):
    """
    Compute omega-squared and partial eta-squared for all fixed effects
    in the LMM via sequential SS (type I), comparing nested models.

    Strategy: for each fixed-effect term, compare the full model to the
    model with that term removed, using ML-fitted log-likelihoods.
    The LL drop is converted to a variance-explained metric.

    Returns a DataFrame with effect sizes per term.
    """
    import re

    # Parse terms from formula
    rhs = formula_full.split('~')[1].strip()
    terms = [t.strip() for t in rhs.split('+')]
    # Expand A*B → A, B, A:B  (keep interaction terms as-is for removal)
    expanded = []
    for t in terms:
        if '*' in t:
            a, b = [x.strip() for x in t.split('*')]
            expanded.extend([a, b, f'{a}:{b}'])
        else:
            expanded.append(t)
    # Deduplicate, preserve order
    seen = set()
    unique_terms = []
    for t in expanded:
        if t not in seen:
            unique_terms.append(t)
            seen.add(t)

    # Total variance in outcome
    y = df_data[outcome].values
    ss_total = np.sum((y - y.mean())**2)
    n_obs    = len(y)

    # Full model fitted with ML
    m_full = fit_lmm(f'{outcome} ~ ' + rhs.replace('*', '+').replace('  ', ' '),
                     df_data, reml=False)
    sigma2_resid = float(m_full.scale)
    ms_error     = sigma2_resid  # residual MS ≈ σ²

    results = []
    for term in unique_terms:
        # Build reduced formula (remove this term and all higher-order
        # terms containing it)
        reduced_terms = [
            t for t in unique_terms
            if term not in t   # removes the term and any interaction containing it
        ]
        if not reduced_terms:
            reduced_formula = f'{outcome} ~ 1'
        else:
            reduced_formula = f'{outcome} ~ ' + ' + '.join(reduced_terms)

        m_reduced = fit_lmm(reduced_formula, df_data, reml=False)

        # SS for this term ≈ explained variance added by the term
        # Approximated via -2ΔlogL (chi-squared) × residual variance
        chi2_drop = 2 * (m_full.llf - m_reduced.llf)
        df_term   = max(m_full.df_modelwc - m_reduced.df_modelwc, 1)
        ss_term   = chi2_drop * sigma2_resid      # approximate SS

        # Omega-squared (bias-corrected)
        omega2 = (ss_term - df_term * ms_error) / (ss_total + ms_error)
        omega2 = max(omega2, 0.0)  # clamp to 0

        # Partial eta-squared
        eta2_partial = ss_term / (ss_term + n_obs * ms_error)
        eta2_partial = np.clip(eta2_partial, 0, 1)

        # Cohen's f
        cohens_f = np.sqrt(eta2_partial / max(1 - eta2_partial, 1e-12))

        # LRT p-value
        p = 1 - stats.chi2.cdf(chi2_drop, df=df_term)

        results.append({
            'Term'          : term,
            'df'            : df_term,
            'chi2'          : round(chi2_drop, 2),
            'p_value'       : p,
            'omega2'        : round(omega2, 5),
            'eta2_partial'  : round(eta2_partial, 5),
            'cohens_f'      : round(cohens_f, 4),
            'interpretation': (
                'Large'  if cohens_f >= 0.40 else
                'Medium' if cohens_f >= 0.25 else
                'Small'  if cohens_f >= 0.10 else
                'Negligible'
            )
        })

    return (pd.DataFrame(results)
              .sort_values('omega2', ascending=False)
              .reset_index(drop=True))


print('Effect size function defined.')

In [ ]:
# ── Compute effect sizes for main-effects model ───────────────────────────────
print('Computing effect sizes for main-effects model...')
es_main = compute_effect_sizes(df_sub, formula_main)
print('\n── Effect Sizes: Main Effects ──────────────────────────────────────────')
print(es_main.to_string(index=False))

In [ ]:
# ── Compute effect sizes for two-way interaction model ────────────────────────
print('Computing effect sizes for two-way interaction model...')
es_2way = compute_effect_sizes(df_sub, formula_2way)
print('\n── Effect Sizes: Main Effects + Two-Way Interactions ───────────────────')
print(es_2way.to_string(index=False))

---
## 5. Variance Partitioning (R²)

Nakagawa & Schielzeth's **marginal** and **conditional** R² decompose the total variance into:
- **Marginal R²**: variance explained by fixed effects (pipeline choices)
- **Conditional R²**: fixed + random (pipeline choices + composition difficulty)
- The difference is the composition-level variance share

In [ ]:
def nakagawa_r2(fitted_model, df_data, outcome='logMSE'):
    """
    Compute marginal and conditional R² following Nakagawa & Schielzeth (2013).
    """
    sigma2_f = np.var(fitted_model.fittedvalues)  # variance of fixed-effect predictions
    sigma2_u = float(fitted_model.cov_re.iloc[0, 0])  # random intercept variance
    sigma2_e = float(fitted_model.scale)               # residual variance

    total = sigma2_f + sigma2_u + sigma2_e

    r2_marginal    = sigma2_f / total
    r2_conditional = (sigma2_f + sigma2_u) / total

    return {
        'sigma2_fixed'    : sigma2_f,
        'sigma2_random'   : sigma2_u,
        'sigma2_residual' : sigma2_e,
        'total'           : total,
        'R2_marginal'     : r2_marginal,
        'R2_conditional'  : r2_conditional,
        'R2_random_only'  : r2_conditional - r2_marginal,
    }


r2_main = nakagawa_r2(m_main, df_sub)
r2_2way = nakagawa_r2(m_2way, df_sub)

print('── Nakagawa R² — Main Effects Model ────────────────────────────────────')
for k, v in r2_main.items():
    print(f'  {k:<25}: {v:.4f}')

print('\n── Nakagawa R² — Two-Way Interaction Model ─────────────────────────────')
for k, v in r2_2way.items():
    print(f'  {k:<25}: {v:.4f}')

---
## 6. Diagnostics

Checking model assumptions: normality of residuals and of random effects.

In [ ]:
def plot_diagnostics(fitted_model, title=''):
    resid   = fitted_model.resid.values
    fitted  = fitted_model.fittedvalues.values
    ranef   = fitted_model.random_effects
    u_vals  = np.array([v['Group'] for v in ranef.values()])

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(f'Diagnostics — {title}', fontweight='bold', fontsize=13)

    # 1. Residuals vs Fitted
    idx = rng.choice(len(resid), 10_000, replace=False)
    axes[0].scatter(fitted[idx], resid[idx], alpha=0.15, s=2, color='steelblue')
    axes[0].axhline(0, color='red', linewidth=1)
    axes[0].set_xlabel('Fitted values')
    axes[0].set_ylabel('Residuals')
    axes[0].set_title('Residuals vs Fitted')

    # 2. Q-Q residuals
    stats.probplot(resid[idx], dist='norm', plot=axes[1])
    axes[1].set_title('Q-Q: Residuals')

    # 3. Residual distribution
    axes[2].hist(resid, bins=80, color='steelblue', edgecolor='white', lw=0.3)
    axes[2].set_xlabel('Residual')
    axes[2].set_title('Residual histogram')

    # 4. Q-Q random effects
    stats.probplot(u_vals, dist='norm', plot=axes[3])
    axes[3].set_title('Q-Q: Random effects $u_m$')

    plt.tight_layout()
    plt.show()

    # Shapiro-Wilk on a 5k subsample
    sw_resid = stats.shapiro(rng.choice(resid, 5000, replace=False))
    sw_ranef = stats.shapiro(u_vals[:5000] if len(u_vals) > 5000 else u_vals)
    print(f'  Shapiro-Wilk residuals : W={sw_resid.statistic:.4f}, p={sw_resid.pvalue:.4f}')
    print(f'  Shapiro-Wilk ranef     : W={sw_ranef.statistic:.4f}, p={sw_ranef.pvalue:.4f}')


plot_diagnostics(m_main, 'Main Effects Model')

In [ ]:
plot_diagnostics(m_2way, 'Two-Way Interaction Model')

---
## 7. Visualisation of Effect Sizes

In [ ]:
# ── 7a. Waterfall chart of ω² ─────────────────────────────────────────────────
palette = {'Large': '#2ecc71', 'Medium': '#3498db',
           'Small': '#e67e22', 'Negligible': '#bdc3c7'}

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=False)

for ax, es, title in [
    (axes[0], es_main, 'Main Effects'),
    (axes[1], es_2way, 'Main + Two-Way Interactions')
]:
    colors = [palette[i] for i in es['interpretation']]
    bars = ax.barh(es['Term'][::-1], es['omega2'][::-1],
                   color=colors[::-1], edgecolor='white', height=0.6)
    ax.set_xlabel('ω² (omega-squared)')
    ax.set_title(f'Effect sizes — {title}', fontweight='bold')

    for bar, val in zip(bars, es['omega2'][::-1]):
        ax.text(bar.get_width() + max(es['omega2']) * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{val:.4f}', va='center', fontsize=9)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in palette.items()]
axes[1].legend(handles=legend_elements, title="Cohen's f", loc='lower right')

plt.tight_layout()
plt.show()

In [ ]:
# ── 7b. R² variance partition pie chart ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, r2, title in [
    (axes[0], r2_main, 'Main Effects'),
    (axes[1], r2_2way, 'Main + Interactions')
]:
    sizes  = [r2['R2_marginal'],
               r2['R2_random_only'],
               1 - r2['R2_conditional']]
    labels = [
        f"Fixed effects\n(pipeline choices)\n{r2['R2_marginal']:.1%}",
        f"Random effect\n(composition difficulty)\n{r2['R2_random_only']:.1%}",
        f"Residual noise\n{1 - r2['R2_conditional']:.1%}"
    ]
    colors = ['#3498db', '#2ecc71', '#ecf0f1']
    wedges, _ = ax.pie(sizes, colors=colors, startangle=90,
                        wedgeprops={'edgecolor': 'white', 'linewidth': 2})
    ax.legend(wedges, labels, loc='lower center',
               bbox_to_anchor=(0.5, -0.35), fontsize=9)
    ax.set_title(f'Variance partition\n{title}', fontweight='bold')

plt.suptitle('Nakagawa R² decomposition', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 7c. Fixed effect coefficient forest plot ───────────────────────────────────
coef_df = pd.DataFrame({
    'coef' : m_main.fe_params,
    'se'   : m_main.bse_fe,
}).drop('Intercept').reset_index()
coef_df.columns = ['term', 'coef', 'se']
coef_df['ci_lo'] = coef_df['coef'] - 1.96 * coef_df['se']
coef_df['ci_hi'] = coef_df['coef'] + 1.96 * coef_df['se']
coef_df['factor'] = coef_df['term'].str.extract(r'^([A-Za-z]+)')

factor_colors = {
    'Classifier' : '#e74c3c',
    'Labeling'   : '#3498db',
    'Deconvolver': '#2ecc71',
    'Calibrator' : '#9b59b6',
}

fig, ax = plt.subplots(figsize=(9, max(4, len(coef_df) * 0.45)))
for i, row in coef_df.iterrows():
    color = factor_colors.get(row['factor'], 'grey')
    ax.errorbar(row['coef'], i,
                xerr=[[row['coef'] - row['ci_lo']], [row['ci_hi'] - row['coef']]],
                fmt='o', color=color, capsize=4, markersize=6)

ax.axvline(0, color='grey', linestyle='--', linewidth=1)
ax.set_yticks(range(len(coef_df)))
ax.set_yticklabels(coef_df['term'], fontsize=9)
ax.set_xlabel('Coefficient (log-MSE scale, reference = C1, L1, D1, K1)')
ax.set_title('Fixed Effect Estimates — Main Effects Model\n(95% CI)', fontweight='bold')

from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color=c, label=f, linewidth=0)
                   for f, c in factor_colors.items()]
ax.legend(handles=legend_elements, loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# ── 7d. Interaction profile: Deconvolver × Calibrator ─────────────────────────
cell_means = (df_sub.groupby(['Deconvolver','Calibrator'], observed=True)['logMSE']
                    .mean().reset_index())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Line plot
for deconv, grp in cell_means.groupby('Deconvolver', observed=True):
    ax1.plot(grp['Calibrator'], grp['logMSE'], marker='o',
             label=deconv, linewidth=2)
ax1.set_xlabel('Calibrator')
ax1.set_ylabel('Mean log(MSE)')
ax1.set_title('Deconvolver × Calibrator\nInteraction Profile', fontweight='bold')
ax1.legend(title='Deconvolver', frameon=False)

# Heatmap
pivot = cell_means.pivot('Deconvolver', 'Calibrator', 'logMSE')
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn_r',
            linewidths=0.5, ax=ax2, cbar_kws={'label': 'Mean log(MSE)'})
ax2.set_title('Deconvolver × Calibrator\nHeatmap', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 8. Summary Tables

In [ ]:
# ── 8a. Full effect-size summary table ────────────────────────────────────────
def format_p(p):
    if p < 0.001: return '<0.001'
    return f'{p:.3f}'

summary = es_2way[['Term','df','chi2','p_value','omega2',
                   'eta2_partial','cohens_f','interpretation']].copy()
summary['p_value'] = summary['p_value'].map(format_p)
summary.columns    = ['Term','df','χ²','p','ω²','η²(partial)',"Cohen's f",'Size']

print('═' * 80)
print('  EFFECT SIZE SUMMARY TABLE (Two-Way Interaction Model)')
print('═' * 80)
print(summary.to_string(index=False))
print('═' * 80)

In [ ]:
# ── 8b. Variance partition summary ────────────────────────────────────────────
print('═' * 60)
print('  VARIANCE PARTITION — MAIN EFFECTS MODEL')
print('═' * 60)
print(f"  ICC (composition difficulty) : {icc:.3f}")
print(f"  Marginal R²  (pipeline)      : {r2_main['R2_marginal']:.3f}")
print(f"  Conditional R² (pipe+compos) : {r2_main['R2_conditional']:.3f}")
print(f"  ΔR² (composition only)       : {r2_main['R2_random_only']:.3f}")
print(f"  Unexplained residual         : {1 - r2_main['R2_conditional']:.3f}")
print('═' * 60)

# ── 8c. Factor ranking by omega-squared ───────────────────────────────────────
main_only = es_main[~es_main['Term'].str.contains(':')].copy()
print('\n── Factor ranking by ω² (main effects only) ─────────────────────────────')
print(main_only[['Term','omega2',"cohens_f",'interpretation']].to_string(index=False))

print(f"\n→ Most impactful pipeline choice : {main_only.iloc[0]['Term']}")
print(f"→ Least impactful pipeline choice: {main_only.iloc[-1]['Term']}")

In [ ]:
# ── 8d. Export results ────────────────────────────────────────────────────────
with pd.ExcelWriter('lmm_results.xlsx', engine='openpyxl') as writer:
    es_2way.to_excel(writer, sheet_name='Effect_Sizes', index=False)
    es_main.to_excel(writer, sheet_name='Effect_Sizes_MainOnly', index=False)

    r2_df = pd.DataFrame([
        {'Model': 'Main effects',       **r2_main},
        {'Model': 'Main + Interactions',**r2_2way},
    ])
    r2_df.to_excel(writer, sheet_name='Variance_Partition', index=False)

    coef_df.to_excel(writer, sheet_name='Coefficients', index=False)

print('Results exported to lmm_results.xlsx')

---
## 9. Interpretation Guide

| Metric | What it tells you |
|---|---|
| **ICC** | Fraction of variance due to composition difficulty. High ICC validates the need for the mixed model. |
| **ω² (omega-squared)** | Fraction of total variance attributable to each pipeline factor, bias-corrected. This is your primary ranking metric. |
| **η²_partial** | Variance explained by each factor after accounting for all others. Larger than ω² but less conservative. |
| **Cohen's f** | Standardised effect size. Benchmarks: f ≥ 0.10 small, ≥ 0.25 medium, ≥ 0.40 large. |
| **R²_marginal** | Fraction of total variance explained by all fixed effects combined (pipeline choices). |
| **R²_conditional** | Fixed effects + composition difficulty. The rest is pure noise. |
| **Interaction ω²** | If large, the optimal level of factor A depends on which level of factor B you chose. |

### Decision rules
1. If `ω²(interaction) / ω²(main)` > 0.1 for a pair, report conditional recommendations (e.g., "use D3 when K2, D1 when K4").
2. For the final pipeline selection, choose the level that minimises the marginal mean `log(MSE)` for factors with small interactions, and the cell mean for factors with large interactions.
3. Report both ω² and Cohen's f — ω² is on the variance scale, Cohen's f is more interpretable for readers unfamiliar with the outcome scale.